# KHÓA LUẬN TỐT NGHIỆP: PHÂN LOẠI RÁC THẢI ĐA CẤP VỚI HIERARCHICAL RESNET-50 & HỆ THỐNG AN TOÀN 2 CHỐT CHẶN (5-VIEW TTA)
---
### Cấu Trúc Pipeline Toàn Diện:
1. **Thiết lập cấu hình & Khám phá dữ liệu (80/10/10)**: Chia tập dữ liệu chuẩn xác (`SEED = 42`), ánh xạ phân cấp 4 lớp con (`glass`, `metal`, `paper`, `plastic`) vào 3 siêu lớp vật liệu (`cellulose`, `metal`, `polymer_glass`).
2. **Data Pipeline Quang Học (`ResizeMethod.AREA` $320 \times 320$)**: Tích hợp tăng cường hình học cùng `Mixup` và `CutMix` ($\alpha = 0.2$) và tiền xử lý chuẩn ResNet-50.
3. **Kiến Trúc Hierarchical ResNet-50**: Kết hợp `CosineDense(scale=20.0)` cho nhánh phân loại chính (trọng số $0.70$) và nhánh giám sát siêu lớp vật liệu (trọng số $0.30$).
4. **Huấn Luyện 2 Giai Đoạn**: Stage 1 Warm-up Heads (15 epochs) và Stage 2 Deep Fine-Tuning 35 layers đỉnh (25 epochs với CosineDecay, đóng băng BatchNormalization).
5. **Suy Luận 5-View TTA & Tìm Ngưỡng An Toàn Thống Kê ($\mu - \sigma$)**: Khóa 2 chốt chặn cốt lõi từ tập Validation:
   - **Kỹ thuật 1 (Confidence Gate)**: Độ tự tin $\tau_{\text{conf}} = \mu_{\text{conf}} - \sigma_{\text{conf}}$
   - **Kỹ thuật 2 (Margin Gate)**: Khoảng cách biên Top1 - Top2 $\tau_{\text{margin}} = \mu_{\text{margin}} - \sigma_{\text{margin}}$
6. **Kiểm Thử Trên Tập Test, Báo Cáo Nghiệm Thu & Trực Quan Hóa**: Báo cáo số mẫu tự động gắp, chuyển khay thủ công, lỗi chặn được, lỗi lọt lưới và hiển thị hình ảnh chi tiết.

In [ ]:
# CELL 1 === 1. KHỞI TẠO THƯ VIỆN & THIẾT LẬP CẤU HÌNH ===
import os
import gc
import sys
import glob
import time
import json
import random
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.special import softmax

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet50

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Cấu hình GPU động
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'✅ Đã phát hiện và kích hoạt {len(gpus)} GPU: {[g.name for g in gpus]}')
    except RuntimeError as e:
        print(f'⚠️ Cảnh báo GPU: {e}')
else:
    print('⚠️ Không tìm thấy GPU, hệ thống sẽ chạy trên CPU.')

# Thiết lập Seed cố định đảm bảo tính tái lập
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Siêu tham số chuẩn từ mytrashnet-resnet50-retrain-tta-safety
IMG_SIZE = (320, 320)
BATCH_SIZE = 32
EPOCHS_STAGE1 = 15
EPOCHS_STAGE2 = 25
MIXUP_ALPHA = 0.2

STAGE1_CHECKPOINT = 'best_trashnet_resnet50_stage1.keras'
STAGE2_CHECKPOINT = 'best_trashnet_resnet50_retrained.keras'

print('✅ Đã thiết lập xong môi trường và siêu tham số cho Hierarchical ResNet-50!')


In [ ]:
# CELL 2 === 2. TẬP DỮ LIỆU ĐỐI SÁNH & NẠP TẬP TEST ĐỘC LẬP 10% (CHÍNH QUY 100%) ===
import os
import glob
import zipfile
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    SEED
except NameError:
    SEED = 42

TARGET_CLASSES = ['glass', 'metal', 'paper', 'plastic']

def extract_ground_truth_label(filename, fallback_label=None):
    """
    Trích xuất nhãn thực tế chuẩn (Ground Truth) từ tên file ảnh.
    """
    fn = os.path.basename(str(filename)).lower()
    for c in TARGET_CLASSES:
        if fn.startswith(c) or f'_{c}' in fn or f'{c}_' in fn:
            return c
    for c in TARGET_CLASSES:
        if c in fn:
            return c
    if 'cardboard' in fn:
        return 'paper'
    return fallback_label if fallback_label else 'unlabeled'

def discover_path(candidates, target_name=None):
    """
    Tìm kiếm file hoặc thư mục theo danh sách ưu tiên.
    """
    for p in candidates:
        if p and os.path.exists(p):
            return p
    if target_name:
        target_clean = os.path.basename(target_name).lower()
        search_roots = ['/kaggle/input', '/kaggle/working', '.']
        for root in search_roots:
            if os.path.exists(root):
                for dirpath, dirnames, filenames in os.walk(root):
                    for f in filenames:
                        if f.lower() == target_clean:
                            return os.path.join(dirpath, f)
                    for d in dirnames:
                        d_norm = d.lower().replace('-', '_')
                        t_norm = target_clean.replace('-', '_')
                        if d.lower() == target_clean or d_norm == t_norm or t_norm in d_norm:
                            return os.path.join(dirpath, d)
    return None

def find_class_root(base_dir):
    """
    Tìm thư mục chứa trực tiếp 4 thư mục nhãn TARGET_CLASSES:
    ['glass', 'metal', 'paper', 'plastic'].
    """
    if not base_dir or not os.path.exists(base_dir):
        return None
    if all(os.path.isdir(os.path.join(base_dir, c)) for c in TARGET_CLASSES):
        return base_dir
    for root, dirs, _ in os.walk(base_dir):
        if all(c in dirs for c in TARGET_CLASSES):
            return root
    return None

# ==============================================================================
# BƯỚC 1: NẠP TẬP TEST ĐỘC LẬP 10% (CÔ LẬP HOÀN TOÀN TỪ ĐẦU - ZERO LEAKAGE)
# ==============================================================================
test_dir_candidates = [
    '/kaggle/input/datasets/vminhkhoi/test-10-percent/MyTrashNet_Test_10Percent',
    '/kaggle/input/test-10-percent/MyTrashNet_Test_10Percent',
    '/kaggle/input/datasets/vminhkhoi/test-10-percent',
    '/kaggle/input/test-10-percent',
    '/kaggle/input/datasets/vminhkhoi/mytrashnet-test-10percent/MyTrashNet_Test_10Percent',
    '/kaggle/input/mytrashnet-test-10percent/MyTrashNet_Test_10Percent',
    '/kaggle/input/datasets/vminhkhoi/mytrashnet-test-10percent',
    '/kaggle/input/mytrashnet-test-10percent',
    '/kaggle/input/MyTrashNet_Test_10Percent',
    'MyTrashNet_Test_10Percent',
    './MyTrashNet_Test_10Percent',
    '../MyTrashNet_Test_10Percent',
    'resnet50/MyTrashNet_Test_10Percent',
    'ConvNeXtTiny/MyTrashNet_Test_10Percent',
    'c:/Users/HP/Downloads/KLTN (code)/MyTrashNet_Test_10Percent'
]

dir_test_raw = discover_path(test_dir_candidates, target_name='MyTrashNet_Test_10Percent')
if not dir_test_raw:
    dir_test_raw = discover_path(test_dir_candidates, target_name='test-10-percent')

# Tự động giải nén nếu dữ liệu test ở dạng file .zip
if dir_test_raw and dir_test_raw.endswith('.zip'):
    extract_test_dir = '/kaggle/working/MyTrashNet_Test_10Percent'
    if not os.path.exists(extract_test_dir):
        print(f"📦 Đang giải nén tập Test: '{dir_test_raw}'...")
        os.makedirs(extract_test_dir, exist_ok=True)
        with zipfile.ZipFile(dir_test_raw, 'r') as zip_ref:
            zip_ref.extractall(extract_test_dir)
        print("✅ Giải nén tập Test thành công!")
    dir_test_raw = extract_test_dir

dir_test = find_class_root(dir_test_raw)
if dir_test is None:
    raise FileNotFoundError("❌ Không tìm thấy thư mục tập Test độc lập 'MyTrashNet_Test_10Percent' chứa 4 lớp!")

print(f"🎯 Thư mục Tập Test độc lập (10% Benchmark): {dir_test}")

test_records = []
for c in TARGET_CLASSES:
    c_path = os.path.join(dir_test, c)
    if os.path.exists(c_path):
        for f in sorted(os.listdir(c_path)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                test_records.append({
                    'filepath': os.path.join(c_path, f),
                    'filename': f,
                    'label': c,
                    'ground_truth': c
                })

test_df = pd.DataFrame(test_records).reset_index(drop=True)
test_filenames_set = set(test_df['filename'].tolist())
print(f"📖 Đã nạp thành công {len(test_df):,} ảnh thuộc tập Test Benchmark độc lập (10% Gold Standard).")
for c in TARGET_CLASSES:
    cnt = len(test_df[test_df['label'] == c])
    print(f"   * Lớp Test [{c:8s}]: {cnt:4d} ảnh")

# ==============================================================================
# BƯỚC 2: KHÁM PHÁ & NẠP KHO DỮ LIỆU ĐỐI SÁNH RESNET-50
# ==============================================================================
train_dir_candidates = [
    '/kaggle/input/datasets/vminhkhoi/dataset-doisanh',
    '/kaggle/input/dataset-doisanh',
    '/kaggle/input/dataset_doi_sanh',
    '/kaggle/input/datasets/vminhkhoi/dataset-doisanh/dataset_doi_sanh',
    '/kaggle/input/dataset-doisanh/dataset_doi_sanh',
    'dataset_doi_sanh',
    './dataset_doi_sanh',
    '../dataset_doi_sanh',
    '/kaggle/working/dataset_doi_sanh',
    '/kaggle/input/datasets/vminhkhoi/dataset-trashnet-2/MyTrashNet4Class',
    '/kaggle/input/dataset-trashnet-2/MyTrashNet4Class',
    '/kaggle/input/dataset-trashnet-2/MyTrashNet',
    '/kaggle/input/dataset-trashnet-2',
    'MyTrashNet4Class',
    './MyTrashNet4Class',
    '../MyTrashNet4Class',
    'c:/Users/HP/Downloads/KLTN (code)/MyTrashNet4Class'
]

dir_train_raw = discover_path(train_dir_candidates, target_name='dataset-doisanh')
if not dir_train_raw:
    dir_train_raw = discover_path(train_dir_candidates, target_name='dataset_doi_sanh')
if not dir_train_raw:
    dir_train_raw = discover_path(train_dir_candidates, target_name='MyTrashNet4Class')

# Tự động giải nén nếu dữ liệu là file .zip
if dir_train_raw and dir_train_raw.endswith('.zip'):
    extract_target = '/kaggle/working/dataset_train_val'
    if not os.path.exists(extract_target):
        print(f"📦 Đang giải nén tập huấn luyện: '{dir_train_raw}'...")
        os.makedirs(extract_target, exist_ok=True)
        with zipfile.ZipFile(dir_train_raw, 'r') as zip_ref:
            zip_ref.extractall(extract_target)
        print("✅ Giải nén tập huấn luyện hoàn tất!")
    dir_train_raw = extract_target

class_root = find_class_root(dir_train_raw)
train_val_records = []

if class_root:
    print(f"📁 Tìm thấy cấu trúc dữ liệu huấn luyện tại: {class_root}")
    for c in TARGET_CLASSES:
        c_path = os.path.join(class_root, c)
        if os.path.exists(c_path):
            for f in sorted(os.listdir(c_path)):
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    # Tự động lọc bỏ nếu ảnh này thuộc tập Test (Phòng thủ đa tầng)
                    if f not in test_filenames_set:
                        train_val_records.append({
                            'filepath': os.path.join(c_path, f),
                            'filename': f,
                            'label': c
                        })
else:
    raise FileNotFoundError("❌ Không tìm thấy thư mục dữ liệu chứa 4 lớp để huấn luyện ResNet-50!")

df_train_val = pd.DataFrame(train_val_records).drop_duplicates(subset=['filename']).reset_index(drop=True)
print(f"📊 Tổng số ảnh huấn luyện & xác thực sạch (Train/Val Pool): {len(df_train_val):,} ảnh")

# ==============================================================================
# BƯỚC 3: PHÂN CHIA TRAIN (80% TỔNG THỂ) VÀ VALIDATION (10% TỔNG THỂ)
# ==============================================================================
# Tập train/val sạch có ~7,254 ảnh (chiếm 90% tổng dữ liệu).
# Để đạt tỉ lệ chuẩn 80% Train và 10% Val của toàn bài toán: val_size = 1/9 ≈ 0.11111
train_df, val_df = train_test_split(
    df_train_val, 
    test_size=(1.0 / 9.0), 
    random_state=SEED, 
    stratify=df_train_val['label']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

N_TOTAL = len(train_df) + len(val_df) + len(test_df)
print(f"\n🎯 PHÂN CHIA DỮ LIỆU ĐỐI SÁNH RESNET-50 (Chuẩn hóa 80/10/10 trên {N_TOTAL:,} ảnh):")
print(f" - Train      : {len(train_df):,} ảnh ({len(train_df)/N_TOTAL*100:.1f}%) [Huấn luyện ResNet-50]")
print(f" - Validation : {len(val_df):,} ảnh ({len(val_df)/N_TOTAL*100:.1f}%)")
print(f" - Test       : {len(test_df):,} ảnh ({len(test_df)/N_TOTAL*100:.1f}%) [Đề thi chung MyTrashNet_Test_10Percent]")

# ==============================================================================
# BƯỚC 4: KIỂM ĐỊNH AN TOÀN TUYỆT ĐỐI (ZERO DATA LEAKAGE)
# ==============================================================================
train_test_overlap = set(train_df['filename']).intersection(test_filenames_set)
val_test_overlap = set(val_df['filename']).intersection(test_filenames_set)

if len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("🛡️ KIỂM ĐỊNH AN TOÀN: 0% RÒ RỈ DỮ LIỆU (Zero Data Leakage). Tập Test hoàn toàn độc lập 100%!")
else:
    raise ValueError(f"❌ CẢNH BÁO: Phát hiện {len(train_test_overlap)} ảnh Train và {len(val_test_overlap)} ảnh Val trùng với Test!")

# ==============================================================================
# BƯỚC 5: THIẾT LẬP BỘ ÁNH XẠ NHÃN & SIÊU LỚP
# ==============================================================================
sub_labels = sorted(TARGET_CLASSES)
num_sub_classes = len(sub_labels)
class_labels = sub_labels
sub_to_idx = {label: i for i, label in enumerate(sub_labels)}
idx_to_sub = {i: label for i, label in enumerate(sub_labels)}

SUPER_MAP = {
    'cardboard': 'cellulose',
    'paper':     'cellulose',
    'glass':     'polymer_glass',
    'plastic':   'polymer_glass',
    'metal':     'metal'
}

sub_to_super_name = {c: SUPER_MAP.get(c, c) for c in sub_labels}
super_labels = sorted(list(set(sub_to_super_name.values())))
num_super_classes = len(super_labels)
super_to_idx = {name: i for i, name in enumerate(super_labels)}
idx_to_super = {i: name for i, name in enumerate(super_labels)}
SUB_TO_SUPER = {sub_to_idx[c]: super_to_idx[sub_to_super_name[c]] for c in sub_labels}

print(f"\n✅ Cấu hình phân cấp Siêu lớp ({num_super_classes} siêu nhóm): khớp hoàn toàn kiến trúc Hierarchical ResNet-50!")


In [ ]:
# CELL 3 === 3. THIẾT LẬP DATA PIPELINE AREA RESAMPLING (320x320) & MIXUP/CUTMIX ===
geometric_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    layers.RandomRotation(factor=(-0.25, 0.25), fill_mode='nearest', interpolation='bilinear', seed=SEED),
    layers.RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2), fill_mode='nearest', interpolation='bilinear', seed=SEED),
    layers.RandomContrast(factor=0.2, seed=SEED),
], name='geometric_augmentation')

def sample_beta(alpha, shape=()):
    gamma1 = tf.random.gamma(shape, alpha, 1.0)
    gamma2 = tf.random.gamma(shape, alpha, 1.0)
    return gamma1 / (gamma1 + gamma2)

def apply_mixup(images, sub_labels, super_labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    indices = tf.random.shuffle(tf.range(batch_size))
    lam = sample_beta(alpha)
    img_shuffled = tf.gather(images, indices)
    sub_shuffled = tf.gather(sub_labels, indices)
    sup_shuffled = tf.gather(super_labels, indices)
    mixed_img = lam * images + (1.0 - lam) * img_shuffled
    mixed_sub = lam * sub_labels + (1.0 - lam) * sub_shuffled
    mixed_sup = lam * super_labels + (1.0 - lam) * sup_shuffled
    return mixed_img, mixed_sub, mixed_sup

def apply_cutmix(images, sub_labels, super_labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    indices = tf.random.shuffle(tf.range(batch_size))
    img_shuffled = tf.gather(images, indices)
    sub_shuffled = tf.gather(sub_labels, indices)
    sup_shuffled = tf.gather(super_labels, indices)
    lam = sample_beta(alpha)
    cut_ratio = tf.sqrt(1.0 - lam)
    img_h = tf.cast(tf.shape(images)[1], tf.float32)
    img_w = tf.cast(tf.shape(images)[2], tf.float32)
    cut_h = tf.cast(img_h * cut_ratio, tf.int32)
    cut_w = tf.cast(img_w * cut_ratio, tf.int32)
    cx = tf.random.uniform([], 0, tf.shape(images)[2], dtype=tf.int32)
    cy = tf.random.uniform([], 0, tf.shape(images)[1], dtype=tf.int32)
    ymin = tf.clip_by_value(cy - cut_h // 2, 0, tf.shape(images)[1])
    xmin = tf.clip_by_value(cx - cut_w // 2, 0, tf.shape(images)[2])
    ymax = tf.clip_by_value(cy + cut_h // 2, 0, tf.shape(images)[1])
    xmax = tf.clip_by_value(cx + cut_w // 2, 0, tf.shape(images)[2])
    mask = tf.pad(
        tf.ones([ymax - ymin, xmax - xmin, tf.shape(images)[3]], dtype=tf.float32),
        [[ymin, tf.shape(images)[1] - ymax], [xmin, tf.shape(images)[2] - xmax], [0, 0]]
    )
    mask = tf.expand_dims(mask, 0)
    mixed_img = images * (1.0 - mask) + img_shuffled * mask
    actual_lam = 1.0 - tf.cast((ymax - ymin) * (xmax - xmin), tf.float32) / (img_h * img_w)
    mixed_sub = actual_lam * sub_labels + (1.0 - actual_lam) * sub_shuffled
    mixed_sup = actual_lam * super_labels + (1.0 - actual_lam) * sup_shuffled
    return mixed_img, mixed_sub, mixed_sup

def batch_mixup_cutmix(images, labels):
    images = tf.cast(images, tf.float32)
    sub_l = tf.cast(labels['sub_class'], tf.float32)
    sup_l = tf.cast(labels['super_class'], tf.float32)
    coin = tf.random.uniform([])
    m_img, m_sub, m_sup = tf.cond(
        coin < 0.5,
        lambda: apply_mixup(images, sub_l, sup_l, alpha=MIXUP_ALPHA),
        lambda: apply_cutmix(images, sub_l, sup_l, alpha=MIXUP_ALPHA)
    )
    return m_img, {'sub_class': m_sub, 'super_class': m_sup}

def create_area_dataset(dataframe, is_training=False, batch_size=32):
    filepaths = dataframe['filepath'].values
    sub_indices = dataframe['label'].map(sub_to_idx).values
    super_indices = np.array([SUB_TO_SUPER[idx] for idx in sub_indices])

    sub_onehot = tf.keras.utils.to_categorical(sub_indices, num_classes=num_sub_classes).astype(np.float32)
    super_onehot = tf.keras.utils.to_categorical(super_indices, num_classes=num_super_classes).astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((
        filepaths,
        {'sub_class': sub_onehot, 'super_class': super_onehot}
    ))

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)

    def parse_img(filename, labels):
        img_bytes = tf.io.read_file(filename)
        img = tf.image.decode_jpeg(img_bytes, channels=3)
        img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.AREA, antialias=True)
        img = tf.cast(img, tf.float32)
        return img, labels

    dataset = dataset.map(parse_img, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)

    if is_training:
        dataset = dataset.map(
            lambda x, y: (preprocess_resnet50(geometric_augmentation(x, training=True)), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
        dataset = dataset.map(batch_mixup_cutmix, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = dataset.map(
            lambda x, y: (preprocess_resnet50(x), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    dataset.filepaths = list(filepaths)
    dataset.classes = list(sub_indices)
    dataset.super_classes = list(super_indices)
    return dataset

train_generator = create_area_dataset(train_df, is_training=True, batch_size=BATCH_SIZE)
val_generator = create_area_dataset(val_df, is_training=False, batch_size=BATCH_SIZE)
test_generator = create_area_dataset(test_df, is_training=False, batch_size=BATCH_SIZE)

print(f'✅ Đã khởi tạo xong Data Pipeline Area Resampling ({IMG_SIZE[0]}x{IMG_SIZE[1]}) cho ResNet-50!')


In [ ]:
# CELL 4 === 4. XÂY DỰNG KIẾN TRÚC MẠNG ĐA TÁC VỤ PHÂN CẤP HIERARCHICAL RESNET-50 ===
try:
    register_serializable = tf.keras.utils.register_keras_serializable
except AttributeError:
    import keras
    register_serializable = keras.saving.register_keras_serializable

@register_serializable(package='CustomLayers', name='CosineDense')
class CosineDense(layers.Layer):
    def __init__(self, units, scale=20.0, **kwargs):
        super(CosineDense, self).__init__(**kwargs)
        self.units = units
        self.scale = scale

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )

    def call(self, inputs):
        norm_inputs = tf.nn.l2_normalize(inputs, axis=-1)
        norm_kernel = tf.nn.l2_normalize(self.kernel, axis=0)
        cosine_sim = tf.matmul(norm_inputs, norm_kernel)
        return self.scale * cosine_sim

    def get_config(self):
        config = super(CosineDense, self).get_config()
        config.update({'units': self.units, 'scale': self.scale})
        return config

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name='input_image')
base_model = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
base_model.trainable = False

x = base_model.output
gap = layers.GlobalAveragePooling2D(name='gap')(x)
gmp = layers.GlobalMaxPooling2D(name='gmp')(x)
shared_features = layers.concatenate([gap, gmp], name='concat_pooling')

shared_features = layers.Dense(1024, activation=None, name='shared_dense')(shared_features)
shared_features = layers.LayerNormalization(name='shared_ln')(shared_features)
shared_features = layers.Activation('gelu', name='shared_gelu')(shared_features)
shared_features = layers.Dropout(0.3, name='shared_dropout')(shared_features)

norm_features = layers.Lambda(lambda t: tf.nn.l2_normalize(t, axis=-1), output_shape=(1024,), name='norm_features')(shared_features)

sub_logits = CosineDense(num_sub_classes, scale=20.0, name='cosine_logits')(norm_features)
sub_class_output = layers.Activation('softmax', name='sub_class', dtype='float32')(sub_logits)

super_dense = layers.Dense(256, activation='gelu', name='super_dense')(shared_features)
super_dense = layers.Dropout(0.2, name='super_dropout')(super_dense)
super_class_output = layers.Dense(num_super_classes, activation='softmax', name='super_class', dtype='float32')(super_dense)

model = Model(
    inputs=inputs,
    outputs={'sub_class': sub_class_output, 'super_class': super_class_output},
    name='Hierarchical_ResNet50_NewLabels'
)

losses = {
    'sub_class': tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    'super_class': tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
}
loss_weights = {
    'sub_class': 0.70,
    'super_class': 0.30
}
metrics = {
    'sub_class': ['accuracy'],
    'super_class': ['accuracy']
}

optimizer_stage1 = tf.keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4)
model.compile(
    optimizer=optimizer_stage1,
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

print('=' * 80)
print('✅ KHỞI TẠO THÀNH CÔNG MÔ HÌNH HIERARCHICAL RESNET-50:')
print(f' - Tổng số tham số : {model.count_params():,}')
print('=' * 80)


In [ ]:
# CELL 5 === 5. GIAI ĐOẠN 1: HUẤN LUYỆN WARM-UP HEADS (15 EPOCHS) ===
callbacks_stage1 = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_sub_class_loss', mode='min', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_sub_class_loss', mode='min', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=STAGE1_CHECKPOINT, monitor='val_sub_class_loss', mode='min', save_best_only=True, verbose=1)
]

print('=' * 80)
print(f'🚀 [GIAI ĐOẠN 1] Huấn luyện Warm-up Heads ResNet-50 ({EPOCHS_STAGE1} epochs)...')
print('=' * 80)
t_st1 = time.time()
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1
)
print(f'✅ Hoàn tất Giai đoạn 1 trong {round(time.time() - t_st1, 1)}s!')


In [ ]:
# CELL 6 === 6. GIAI ĐOẠN 2: DEEP FINE-TUNING 35 LAYERS ĐỈNH VỚI COSINE DECAY (25 EPOCHS) ===
base_model.trainable = True

unfreeze_layers = 35  # Mở khóa tầng đỉnh conv5 block của ResNet-50
for layer in base_model.layers[:-unfreeze_layers]:
    layer.trainable = False
for layer in base_model.layers[-unfreeze_layers:]:
    # Giữ nguyên BatchNormalization ở chế độ inference để ổn định thống kê
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

unfrozen_cnt = sum(1 for l in base_model.layers if l.trainable)
print(f'🔓 ResNet-50 Fine-Tuning: Đã mở khóa {unfrozen_cnt} layers đỉnh cao nhất (đóng băng BatchNorm)!')

total_steps = len(train_df) // BATCH_SIZE * EPOCHS_STAGE2
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=total_steps,
    alpha=0.01
)

optimizer_stage2 = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

model.compile(
    optimizer=optimizer_stage2,
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_sub_class_loss', mode='min', patience=8, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=STAGE2_CHECKPOINT, monitor='val_sub_class_loss', mode='min', save_best_only=True, verbose=1)
]

print('=' * 80)
print(f'🚀 [GIAI ĐOẠN 2] Deep Fine-Tuning ResNet-50 ({EPOCHS_STAGE2} epochs)...')
print('=' * 80)
t_st2 = time.time()
history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_stage2
)
print(f'✅ Hoàn tất Giai đoạn 2 trong {round(time.time() - t_st2, 1)}s!')


In [ ]:
# CELL 7 === 7. LƯU LỊCH SỬ HUẤN LUYỆN & TRỰC QUAN HÓA ĐỒ THỊ QUÁ TRÌNH HỌC ===
def clean_history(hist):
    return {k: [float(val) for val in v] for k, v in hist.items()}

history_dict = {
    'stage1': clean_history(history_stage1.history),
    'stage2': clean_history(history_stage2.history)
}

with open('training_history_resnet50.json', 'w', encoding='utf-8') as f:
    json.dump(history_dict, f, indent=4)
print('✅ Đã lưu lịch sử huấn luyện vào file training_history_resnet50.json')

# Vẽ đồ thị tiến trình huấn luyện 2 giai đoạn
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

acc_all = history_dict['stage1']['sub_class_accuracy'] + history_dict['stage2']['sub_class_accuracy']
val_acc_all = history_dict['stage1']['val_sub_class_accuracy'] + history_dict['stage2']['val_sub_class_accuracy']
loss_all = history_dict['stage1']['sub_class_loss'] + history_dict['stage2']['sub_class_loss']
val_loss_all = history_dict['stage1']['val_sub_class_loss'] + history_dict['stage2']['val_sub_class_loss']
split_epoch = len(history_dict['stage1']['sub_class_accuracy']) - 1

axes[0].plot(acc_all, label='Train Sub-class Accuracy', color='tab:blue', lw=2)
axes[0].plot(val_acc_all, label='Val Sub-class Accuracy', color='tab:orange', lw=2)
axes[0].axvline(x=split_epoch, color='red', linestyle='--', label='Bắt đầu Fine-tuning (Stage 2)')
axes[0].set_title('Đồ thị Sub-class Accuracy qua các Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_all, label='Train Sub-class Loss', color='tab:blue', lw=2)
axes[1].plot(val_loss_all, label='Val Sub-class Loss', color='tab:orange', lw=2)
axes[1].axvline(x=split_epoch, color='red', linestyle='--', label='Bắt đầu Fine-tuning (Stage 2)')
axes[1].set_title('Đồ thị Sub-class Loss qua các Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('resnet50_training_curves.png', dpi=300)
plt.show()


In [ ]:
# CELL 8 === 8. SUY LUẬN 5-VIEW TTA & TỰ ĐỘNG TÌM BỘ NGƯỠNG AN TOÀN RIÊNG CHO TỪNG LỚP ===

# ==============================================================================
# BƯỚC 0: TRUYỀN ĐƯỜNG DẪN & NẠP MODEL KHI MỞ LẠI MÁY
# ==============================================================================
MODEL_PATH = '/kaggle/input/models/vminhkhoi/resnet50/keras/default/1/best_trashnet_resnet50_retrained.keras'

# Dự phòng nếu chạy trên máy cá nhân hoặc thư mục làm việc hiện tại
if not os.path.exists(MODEL_PATH):
    local_candidates = [
        STAGE2_CHECKPOINT if 'STAGE2_CHECKPOINT' in globals() else 'best_trashnet_resnet50_retrained.keras',
        'best_trashnet_resnet50_retrained.keras',
        './best_trashnet_resnet50_retrained.keras',
        'resnet50/best_trashnet_resnet50_retrained.keras',
        '../best_trashnet_resnet50_retrained.keras'
    ]
    for cand in local_candidates:
        if cand and os.path.exists(cand):
            MODEL_PATH = cand
            break

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy file model tại: '{MODEL_PATH}'. Vui lòng kiểm tra lại đường dẫn!")

print(f"📦 Đang nạp mô hình từ: '{MODEL_PATH}' ({os.path.getsize(MODEL_PATH) / (1024*1024):.2f} MB)...")

try:
    custom_objects = {'CosineDense': CosineDense}
    model = tf.keras.models.load_model(MODEL_PATH, custom_objects=custom_objects, safe_mode=False, compile=False)
    for l in model.layers:
        if hasattr(l, 'function') and hasattr(l.function, '__globals__'):
            l.function.__globals__['tf'] = tf
            l.function.__globals__['tensorflow'] = tf
    print(f"✅ Nạp thành công toàn bộ mô hình ResNet-50 ({model.count_params():,} tham số)!")
except Exception as e:
    print(f"ℹ️ Thử nạp theo dạng weights: {e}")
    model.load_weights(MODEL_PATH)
    print("✅ Đã nạp thành công trọng số vào kiến trúc mô hình!")

sub_logits_model = Model(inputs=model.input, outputs=model.get_layer('cosine_logits').output)

def make_tta_dataset(filepaths, view_idx=0, batch_size=64):
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    def load_and_aug(fpath):
        img_bytes = tf.io.read_file(fpath)
        img = tf.image.decode_jpeg(img_bytes, channels=3)
        img = tf.image.resize(img, IMG_SIZE, method=tf.image.ResizeMethod.AREA, antialias=True)
        img = tf.cast(img, tf.float32)

        h, w = IMG_SIZE
        if view_idx == 0: pass
        elif view_idx == 1: img = tf.image.flip_left_right(img)
        elif view_idx == 2: img = tf.image.rot90(img, k=1)
        elif view_idx == 3: img = tf.image.rot90(img, k=3)
        elif view_idx == 4:
            crop_size = int(h * 0.90)
            offset = (h - crop_size) // 2
            img_c = tf.image.crop_to_bounding_box(img, offset, offset, crop_size, crop_size)
            img = tf.image.resize(img_c, IMG_SIZE, method=tf.image.ResizeMethod.AREA, antialias=True)

        return preprocess_resnet50(img)

    ds = ds.map(load_and_aug, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

print('=' * 105)
print('BƯỚC 1: SUY LUẬN ĐA GÓC NHÌN 5-VIEW TTA TRÊN TẬP VALIDATION...')
print('=' * 105)
val_filepaths = val_df['filepath'].values
val_trues = val_df['label'].map(sub_to_idx).values
N_VAL = len(val_filepaths)

val_views_probs = []
for v in range(5):
    print(f' -> Đang dự đoán View {v+1}/5 trên tập Validation...')
    v_ds = make_tta_dataset(val_filepaths, view_idx=v, batch_size=BATCH_SIZE*2)
    logits = sub_logits_model.predict(v_ds, batch_size=BATCH_SIZE*2, verbose=0)
    probs = softmax(logits, axis=1)
    val_views_probs.append(probs)

val_tta_probs = np.mean(val_views_probs, axis=0)
val_tta_preds = np.argmax(val_tta_probs, axis=1)

val_sorted_probs = np.sort(val_tta_probs, axis=1)
val_max_confs = val_sorted_probs[:, -1]
val_margins = val_sorted_probs[:, -1] - val_sorted_probs[:, -2]

val_acc = np.mean(val_tta_preds == val_trues) * 100.0
val_errors = int(np.sum(val_tta_preds != val_trues))
print(f'\n✅ Validation 5-View TTA Accuracy: {val_acc:.2f}% ({val_errors} lỗi trên {N_VAL} mẫu)')
print('\n--- BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP VALIDATION (5-VIEW TTA) ---')
print(classification_report(val_trues, val_tta_preds, target_names=class_labels, digits=4))

# ==============================================================================
# BƯỚC 2: KHẢO SÁT & TỐI ƯU HÓA NGƯỠNG CHỐT 1 RIÊNG BIỆT CHO TỪNG LỚP (PER-CLASS)
# ==============================================================================
K1_CANDIDATES = [0.8, 1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0]

TAU_CONF_DICT = {}   # Lưu ngưỡng riêng của từng lớp: {class_idx: tau}
BEST_K1_DICT = {}    # Lưu hệ số k1 tốt nhất của từng lớp

print('\n' + '=' * 105)
print('📊 [BẢNG 1] KHẢO SÁT CHỐT 1 (CONFIDENCE GATE) RIÊNG TỪNG LỚP:')
print('=' * 105)

for c_idx, c_name in enumerate(class_labels):
    # Lọc các mẫu dự đoán đúng của riêng lớp c_name để tính mean và std đặc trưng
    mask_correct_c = (val_tta_preds == c_idx) & (val_tta_preds == val_trues)
    confs_c = val_max_confs[mask_correct_c]
    mu_c = float(np.mean(confs_c))
    sigma_c = float(np.std(confs_c))
    
    # Tất cả các mẫu mô hình dự đoán là lớp c_name
    mask_pred_c = (val_tta_preds == c_idx)
    total_pred_c = int(np.sum(mask_pred_c))
    
    print(f"\n🏷️ LỚP [{c_name.upper()}]: (mu = {mu_c*100:.2f}%, sigma = {sigma_c*100:.2f}%, Tổng dự đoán: {total_pred_c} mẫu)")
    print(f"{'Hệ số k1':>10} {'Ngưỡng Conf':>14} {'Lỗi lọt lưới':>15} {'Tỷ lệ gắp lớp':>22}  {'Đánh giá'}")
    print('-' * 100)
    
    class_results = []
    for k in K1_CANDIDATES:
        tau = float(np.clip(mu_c - k * sigma_c, 0.50, 0.99))
        
        # Mẫu dự đoán là c_name và vượt qua ngưỡng tau
        pass_gate = mask_pred_c & (val_max_confs >= tau)
        leaked_errors = int(np.sum(pass_gate & (val_trues != c_idx)))
        picked_samples = int(np.sum(pass_gate))
        pick_rate = (picked_samples / total_pred_c * 100.0) if total_pred_c > 0 else 0.0
        
        class_results.append({
            'k': k,
            'tau': tau,
            'leaked': leaked_errors,
            'picked': picked_samples,
            'total': total_pred_c,
            'rate': pick_rate
        })
        
    # Tiêu chí chọn k1 tối ưu: 
    # 1. Số lỗi lọt lưới nhỏ nhất (ưu tiên 0 lỗi)
    # 2. Tỷ lệ gắp mẫu (Coverage) lớn nhất
    min_err = min(r['leaked'] for r in class_results)
    valid_candidates = [r for r in class_results if r['leaked'] == min_err]
    best_candidate = max(valid_candidates, key=lambda x: x['rate'])
    
    for r in class_results:
        k = r['k']
        tau = r['tau']
        leaked = r['leaked']
        picked = r['picked']
        total = r['total']
        rate = r['rate']
        
        if k == best_candidate['k']:
            status = "⭐ ĐIỂM TỐI ƯU (0 lỗi & gắp tối đa)" if leaked == 0 else "⭐ ĐIỂM TỐI ƯU (Lỗi tối thiểu)"
        elif leaked == 0:
            status = "✅ An toàn (Bắt nhầm mẫu đúng)"
        else:
            status = f"⚠️ Cảnh báo: Lọt {leaked} lỗi sai"
            
        print(f"k1 = {k:<5.1f}   >= {tau*100:>6.2f}%   {leaked:>10d} lỗi   {rate:>6.2f}% ({picked:>3d}/{total:<3d})   {status}")
        
    TAU_CONF_DICT[c_idx] = best_candidate['tau']
    BEST_K1_DICT[c_idx] = best_candidate['k']

# Tổng hợp bộ 4 ngưỡng Chốt 1 đã chốt
print('\n' + '=' * 105)
print('🎯 TỔNG HỢP BỘ 4 NGƯỠNG TỰ TIN (CHỐT 1) THÍCH ỨNG THEO TỪNG LỚP:')
print('=' * 105)
for c_idx, c_name in enumerate(class_labels):
    print(f" * Lớp {c_name:<10s}: k1 = {BEST_K1_DICT[c_idx]:.1f}  -->  Ngưỡng Conf >= {TAU_CONF_DICT[c_idx]*100:.2f}%")

# Tính kết quả Chốt 1 trên toàn bộ tập Validation
val_sample_tau_conf = np.array([TAU_CONF_DICT[p] for p in val_tta_preds])
val_pass_1 = (val_max_confs >= val_sample_tau_conf)
val_p1_cov = np.mean(val_pass_1) * 100.0
val_p1_err = int(np.sum(val_pass_1 & (val_tta_preds != val_trues)))
print('-' * 105)
print(f"👉 Hiệu quả tổng hợp Chốt 1: Tự động gắp {val_p1_cov:.2f}% ({np.sum(val_pass_1)}/{N_VAL} mẫu) | Số lỗi lọt lưới: {val_p1_err} lỗi")

# ==============================================================================
# BƯỚC 3: CỐ ĐỊNH BỘ 4 NGƯỠNG CHỐT 1, KHẢO SÁT PHỐI HỢP CHỐT 2 (MARGIN GATE)
# ==============================================================================
mean_margin, std_margin = float(np.mean(val_margins[val_tta_preds == val_trues])), float(np.std(val_margins[val_tta_preds == val_trues]))
K2_CANDIDATES = [0.8, 1.0, 1.2, 1.5, 1.8, 2.0, 2.2, 2.4, 2.6]

print('\n' + '=' * 105)
print(f"📊 [BẢNG 2] CỐ ĐỊNH BỘ 4 NGƯỠNG CHỐT 1, KHẢO SÁT PHỐI HỢP VỚI CHỐT 2 (MARGIN GATE):")
print(f"   (Thống kê Margin: mu = {mean_margin*100:.2f}%, sigma = {std_margin*100:.2f}%)")
print('=' * 105)
print(f"{'Thử Chốt 2':>15} {'Ngưỡng Margin':>16} {'Lỗi lọt lưới':>15} {'Tỷ lệ gắp kết hợp':>22}  {'Đánh giá khi phối hợp'}")
print('-' * 105)

results_table_2 = []
for k2 in K2_CANDIDATES:
    tau_m = float(np.clip(mean_margin - k2 * std_margin, 0.30, 0.95))
    pass_2 = (val_margins >= tau_m)
    pass_combined = val_pass_1 & pass_2
    
    leaked_combined = int(np.sum(pass_combined & (val_tta_preds != val_trues)))
    picked_combined = int(np.sum(pass_combined))
    cov_combined = (picked_combined / N_VAL) * 100.0
    
    results_table_2.append({
        'k2': k2,
        'tau_m': tau_m,
        'leaked': leaked_combined,
        'picked': picked_combined,
        'cov': cov_combined
    })

# Chọn k2 tối ưu (0 lỗi và gắp nhiều nhất)
min_err_t2 = min(r['leaked'] for r in results_table_2)
valid_t2 = [r for r in results_table_2 if r['leaked'] == min_err_t2]
max_cov_t2 = max(r['cov'] for r in valid_t2)
best_t2 = max(valid_t2, key=lambda x: x['cov'])

for r in results_table_2:
    k2 = r['k2']
    tau_m = r['tau_m']
    leaked = r['leaked']
    picked = r['picked']
    cov = r['cov']
    
    if k2 == best_t2['k2']:
        status = "⭐ ĐIỂM TỐI ƯU (Tự động chọn: An toàn cao nhất & gắp tối đa)"
    elif leaked == 0 and cov < max_cov_t2:
        status = "✅ An toàn (Bắt nhầm nhiều mẫu đúng)"
    elif leaked == 0:
        status = f"✅ An toàn phối hợp (Gắp {cov:.2f}%)"
    else:
        status = f"⚠️ Cảnh báo: Lọt {leaked} lỗi sai"
        
    print(f"k2 = {k2:<5.1f}   >= {tau_m*100:>6.2f}%   {leaked:>10d} lỗi   {cov:>6.2f}% ({picked:>3d}/{N_VAL:<3d})   {status}")

TAU_MARGIN = best_t2['tau_m']
BEST_K2 = best_t2['k2']

print('\n' + '=' * 105)
print('🏆 KẾT LUẬN: BỘ 2 CHỐT CHẶN AN TOÀN ĐƯỢC CHỐT HẠ CHO TOÀN BỘ HỆ THỐNG:')
print('=' * 105)
print(f"📌 CHỐT 1 (CONFIDENCE GATE - THEO TỪNG LỚP):")
for c_idx, c_name in enumerate(class_labels):
    print(f"   - Lớp {c_name:<10s}: Conf >= {TAU_CONF_DICT[c_idx]*100:.2f}% (k1 = {BEST_K1_DICT[c_idx]:.1f})")
print(f"📌 CHỐT 2 (MARGIN GATE - CHUNG)            : Margin >= {TAU_MARGIN*100:.2f}% (k2 = {BEST_K2:.1f})")
print('-' * 105)

val_pass_final = val_pass_1 & (val_margins >= TAU_MARGIN)
val_final_cov = np.mean(val_pass_final) * 100.0
val_final_err = int(np.sum(val_pass_final & (val_tta_preds != val_trues)))
val_final_acc = ((np.sum(val_pass_final) - val_final_err) / np.sum(val_pass_final) * 100.0) if np.sum(val_pass_final) > 0 else 0.0
caught_err_val = val_errors - val_final_err

print(f"Hiệu năng kiểm chứng trên tập Validation:")
print(f"  * Tỷ lệ tự động gắp an toàn   : {val_final_cov:.2f}% ({np.sum(val_pass_final)} / {N_VAL} mẫu)")
print(f"  * Độ chính xác khi tự động gắp: {val_final_acc:.2f}%")
print(f"  * Số lỗi bắt giữ (tách kiểm)  : {caught_err_val} / {val_errors} lỗi ({caught_err_val/val_errors*100.0:.2f}%)")
print(f"  * Số lỗi lọt lưới ra ngoài    : {val_final_err} lỗi (An toàn tuyệt đối)")
print('=' * 105)


In [ ]:
# CELL 9 === 9. VẬN HÀNH BỘ LỌC 5-VIEW TTA & CƠ CHẾ BẢO LÃNH ĐỒNG THUẬN TTA TRÊN TẬP TEST ===
print('=' * 105)
print('BƯỚC 2: SUY LUẬN 5-VIEW TTA & VẬN HÀNH BỘ LỌC ĐIỂM BÙ TRỪ + BẢO LÃNH TTA TRÊN TẬP TEST...')
print('=' * 105)

test_filepaths = test_df['filepath'].values
test_trues = test_df['label'].map(sub_to_idx).values
test_filenames = test_df['filename'].values
N_test = len(test_filepaths)

# ==============================================================================
# 1. SUY LUẬN ĐA GÓC NHÌN 5-VIEW TTA TRÊN TẬP TEST
# ==============================================================================
test_views_probs = []
for v in range(5):
    print(f' -> Đang suy luận View {v+1}/5 trên tập Test...')
    v_ds = make_tta_dataset(test_filepaths, view_idx=v, batch_size=BATCH_SIZE*2)
    logits = sub_logits_model.predict(v_ds, batch_size=BATCH_SIZE*2, verbose=0)
    probs = softmax(logits, axis=1)
    test_views_probs.append(probs)

# Tổng hợp kết quả xác suất trung bình 5-View TTA
test_tta_probs = np.mean(test_views_probs, axis=0)
test_final_preds = np.argmax(test_tta_probs, axis=1)

test_sorted_probs = np.sort(test_tta_probs, axis=1)
test_max_confs = test_sorted_probs[:, -1]
test_margins = test_sorted_probs[:, -1] - test_sorted_probs[:, -2]

# Độ đồng thuận giữa 5 góc nhìn TTA (Consensus Stability)
test_views_preds = np.argmax(np.array(test_views_probs), axis=2)
test_tta_stability = np.mean(test_views_preds == test_final_preds[None, :], axis=0)

# Báo cáo tổng thể mô hình gốc (trước khi qua bộ lọc)
test_raw_acc = np.mean(test_final_preds == test_trues) * 100.0
total_errors = int(np.sum(test_final_preds != test_trues))
print(f'\n📊 Độ chính xác 5-View TTA gốc trên tập Test: {test_raw_acc:.2f}% ({total_errors} lỗi trên {N_test} mẫu)')
print('\n--- CLASSIFICATION REPORT (TEST SET - 5-VIEW TTA) ---')
print(classification_report(test_trues, test_final_preds, target_names=class_labels, digits=4))

# Vẽ Confusion Matrix
test_cm = confusion_matrix(test_trues, test_final_preds)
plt.figure(figsize=(7, 6), dpi=150)
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title(f'Confusion Matrix - ResNet-50 Test Set 5-View TTA (Acc: {test_raw_acc:.2f}%)', fontweight='bold')
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.tight_layout()
plt.savefig('confusion_matrix_resnet50_test_tta.png', dpi=300)
plt.show()

# ==============================================================================
# 2. VẬN HÀNH BỘ LỌC AN TOÀN ĐA CẤP (3-TIER ADAPTIVE SAFETY FILTER)
# ==============================================================================
# THIẾT LẬP CÁC NGƯỠNG ĐIỀU KHIỂN:
W_CONF = 0.50             # Trọng số độ tự tin Chốt 1
W_MARGIN = 0.50           # Trọng số biên phân tách Chốt 2
CONF_FLOOR_TIER1 = 0.70   # Ngưỡng sàn cho Cấp 1 (S_safety >= 1.0)
CONF_FLOOR_TIER2 = 0.65   # Ngưỡng sàn cho Cấp 2 (S_safety >= 0.90 kèm bảo lãnh TTA 100%)
TOLERANCE_SCORE = 0.90    # Dung sai an toàn 10% (S_safety >= 90%)

records_test = []
count_standard = 0        # Đếm số mẫu đạt chuẩn cơ sở
count_rescued_c1 = 0      # Đếm số mẫu cứu nguy Cấp 1 (Bù trừ S >= 1.0)
count_rescued_c2 = 0      # Đếm số mẫu cứu nguy Cấp 2 (Bảo lãnh TTA 100% & S >= 0.90)

for i in range(N_test):
    pred_c = test_final_preds[i]
    true_c = test_trues[i]
    top1_c = float(test_max_confs[i])
    m_val = float(test_margins[i])
    tta_stab = float(test_tta_stability[i])

    # Lấy ngưỡng riêng của từng lớp từ Cell 8
    tau_conf_c = TAU_CONF_DICT[pred_c]

    # Tính điểm an toàn chuẩn hóa của từng chốt
    s_conf = top1_c / tau_conf_c
    s_margin = m_val / TAU_MARGIN
    composite_safety_score = (W_CONF * s_conf) + (W_MARGIN * s_margin)

    # 1. ĐIỀU KIỆN CẤP 1: Vượt chuẩn cả 2 chốt độc lập (Hard Gates)
    pass_tier1_standard = (top1_c >= tau_conf_c) and (m_val >= TAU_MARGIN)

    # 2. ĐIỀU KIỆN CẤP 2: Đạt chuẩn điểm an toàn bù trừ (Composite Soft Score >= 1.0)
    pass_tier2_composite = (composite_safety_score >= 1.0) and (top1_c >= CONF_FLOOR_TIER1)

    # 3. ĐIỀU KIỆN CẤP 3: Cửa phụ Bảo Lãnh Đồng Thuận TTA 100% (S >= 0.90 & TTA=100% & Conf >= 65%)
    pass_tier3_consensus = (composite_safety_score >= TOLERANCE_SCORE) and (tta_stab >= 0.999) and (top1_c >= CONF_FLOOR_TIER2)

    # Thu thập lý do nếu trượt chuẩn cơ sở
    reasons = []
    if top1_c < tau_conf_c:
        reasons.append(f'Chốt 1: Tự tin thấp lớp {class_labels[pred_c]} ({top1_c*100:.1f}% < {tau_conf_c*100:.1f}%)')
    if m_val < TAU_MARGIN:
        reasons.append(f'Chốt 2: Biên Margin hẹp ({m_val*100:.1f}% < {TAU_MARGIN*100:.1f}%)')

    # RA QUYẾT ĐỊNH PHÂN LUỒNG:
    if pass_tier1_standard:
        decision = 'Tu_Dong_Gap'
        reason = f'Đạt chuẩn cơ sở 5-View TTA (Chốt 1: {s_conf:.2f}x | Chốt 2: {s_margin:.2f}x)'
        count_standard += 1
    elif pass_tier2_composite:
        decision = 'Tu_Dong_Gap'
        reason = f'Cứu nguy bù trừ an toàn (Điểm: {composite_safety_score*100:.1f}% >= 100%)'
        count_rescued_c1 += 1
    elif pass_tier3_consensus:
        decision = 'Tu_Dong_Gap'
        reason = f'Cứu nguy bảo lãnh TTA 100% (Điểm: {composite_safety_score*100:.1f}% >= 90% | Đồng thuận: 100%)'
        count_rescued_c2 += 1
    else:
        decision = 'Chuyen_Khay_Thu_Cong'
        reason = ' | '.join(reasons)

    records_test.append({
        'filename': test_filenames[i],
        'filepath': test_filepaths[i],
        'true_label': class_labels[true_c],
        'pred_label': class_labels[pred_c],
        'confidence': f'{top1_c*100:.2f}%',
        'margin': f'{m_val*100:.2f}%',
        'tta_stability': f'{tta_stab*100:.0f}%',
        'composite_score': f'{composite_safety_score*100:.2f}%',
        'decision': decision,
        'reason': reason
    })

df_results = pd.DataFrame(records_test)

auto_samples = df_results[df_results['decision'] == 'Tu_Dong_Gap']
manual_samples = df_results[df_results['decision'] == 'Chuyen_Khay_Thu_Cong']

caught_errors = manual_samples[manual_samples['true_label'] != manual_samples['pred_label']]
leaked_errors = auto_samples[auto_samples['true_label'] != auto_samples['pred_label']]

auto_acc = np.mean(auto_samples['true_label'] == auto_samples['pred_label']) * 100.0 if len(auto_samples) > 0 else 0.0
caught_rate = (len(caught_errors) / total_errors * 100.0) if total_errors > 0 else 100.0
leaked_rate = (len(leaked_errors) / total_errors * 100.0) if total_errors > 0 else 0.0

# ==============================================================================
# 3. XUẤT BÁO CÁO NGHIỆM THU CHI TIẾT
# ==============================================================================
print('\n' + '=' * 105)
print('🏆 BÁO CÁO NGHIỆM THU HỆ THỐNG AN TOÀN 5-VIEW TTA & ĐA CẤP BÙ TRỪ TRÊN TẬP TEST (RESNET-50):')
print('=' * 105)
print(f'Số mẫu tự động gắp an toàn             : {len(auto_samples)} / {N_test} ({len(auto_samples)/N_test*100:.2f}%)')
print(f'  * Vượt chuẩn cơ sở cả 2 chốt          : {count_standard} mẫu')
print(f'  * Cứu nguy Cấp 1 (Điểm bù trừ >= 100%): {count_rescued_c1} mẫu')
print(f'  * Cứu nguy Cấp 2 (Bảo lãnh TTA 100%)  : {count_rescued_c2} mẫu')
print(f'ĐỘ CHÍNH XÁC KHI GẮP TỰ ĐỘNG           : {auto_acc:.2f}% ({len(auto_samples) - len(leaked_errors)} / {len(auto_samples)})')
print(f'Số mẫu chuyển sang khay thủ công       : {len(manual_samples)} / {N_test} ({len(manual_samples)/N_test*100:.2f}%)')
print('-' * 105)
print(f'Tổng số lỗi sai thực tế trên tập Test  : {total_errors} mẫu (100.00%)')
print(f'SỐ LỖI ĐÃ TÓM GỌN THÀNH CÔNG           : {len(caught_errors)} / {total_errors} ({caught_rate:.2f}%)')
print(f'SỐ LỖI CÒN BỊ LỌT LƯỚI (CẢNH BÁO)      : {len(leaked_errors)} / {total_errors} ({leaked_rate:.2f}%)')
print('=' * 105)

df_results.to_csv('safety_classification_resnet50_report.csv', index=False, encoding='utf-8-sig')
print("✅ Đã lưu toàn bộ báo cáo chi tiết vào file: 'safety_classification_resnet50_report.csv'.")


In [ ]:
# CELL 10 === 10. TRỰC QUAN HÓA HÌNH ẢNH CÁC MẪU ĐOÁN ĐÚNG, LỖI ĐÃ CHẶN & LỖI LỌT LƯỚI ===
def plot_inspected_samples(samples_df, title, num_show=6, title_color='#d62728', save_name='samples_inspected.png'):
    if len(samples_df) == 0:
        print(f'ℹ️ Không có mẫu nào để hiển thị cho: {title}')
        return

    num_show = min(num_show, len(samples_df))
    cols = 3
    rows = int(np.ceil(num_show / cols))

    plt.figure(figsize=(15, 4.5 * rows), dpi=150)
    plt.suptitle(title, fontsize=13, fontweight='bold', y=0.98)

    for idx in range(num_show):
        row = samples_df.iloc[idx]
        img_path = row['filepath']
        img = None
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if img is None:
            img = np.zeros((320, 320, 3), dtype=np.uint8)

        ax = plt.subplot(rows, cols, idx + 1)
        ax.imshow(img)
        ax.axis('off')

        caption = (
            f"Tên: {row['filename']}\n"
            f"Thực tế: {row['true_label'].upper()} | Đoán: {row['pred_label'].upper()} (Conf: {row['confidence']})\n"
            f"Biên Margin: {row['margin']} | Đồng thuận 5-View TTA: {row['tta_stability']}\n"
            f"Kết luận: {row['reason']}"
        )
        ax.set_title(caption, fontsize=8.5, color=title_color, fontweight='bold', pad=4)

    plt.tight_layout()
    plt.savefig(save_name, dpi=300)
    plt.show()

# Tách nhóm các mẫu Chuyển khay thủ công: Đoán SAI (Cứu nguy) vs Đoán ĐÚNG (Thận trọng)
manual_wrong = manual_samples[manual_samples['true_label'] != manual_samples['pred_label']]
manual_correct = manual_samples[manual_samples['true_label'] == manual_samples['pred_label']]

print('=' * 90)
print(f"📊 THỐNG KÊ KHAY THỦ CÔNG: Tổng {len(manual_samples)} mẫu = {len(manual_wrong)} mẫu sai (cứu nguy) + {len(manual_correct)} mẫu đúng (thận trọng)")
print('=' * 90)

# 1. HIỂN THỊ 6 MẪU CHUYỂN THỦ CÔNG DO ĐOÁN SAI (LỖI ĐÃ TÓM GỌN THÀNH CÔNG VÀO KHAY)
if len(manual_wrong) > 0:
    plot_inspected_samples(
        manual_wrong.head(6),
        title=f'RESNET-50: 6 MẪU LỖI SAI ĐƯỢC CHẶN VÀO KHAY THỦ CÔNG (HIỂN THỊ 6/{len(manual_wrong)} LỖI)',
        num_show=6,
        title_color='#d62728',
        save_name='mau_sai_chuyen_thu_cong_resnet50_tta.png'
    )

# 2. HIỂN THỊ 6 MẪU CHUYỂN THỦ CÔNG DÙ ĐOÁN ĐÚNG (CẢNH BÁO AN TOÀN DO CHƯA ĐẠT CHUẨN)
if len(manual_correct) > 0:
    plot_inspected_samples(
        manual_correct.sample(min(6, len(manual_correct)), random_state=SEED),
        title=f'RESNET-50: 6 MẪU ĐOÁN ĐÚNG NHƯNG CHUYỂN THỦ CÔNG CẨN TRỌNG (HIỂN THỊ 6/{len(manual_correct)} MẪU)',
        num_show=6,
        title_color='#1f77b4',
        save_name='mau_dung_chuyen_thu_cong_resnet50_tta.png'
    )

# 3. HIỂN THỊ CÁC MẪU LỖI CÒN BỊ LỌT LƯỚI (NẾU CÓ)
if len(leaked_errors) > 0:
    plot_inspected_samples(
        leaked_errors,
        title=f'CẢNH BÁO: CÁC MẪU LỖI CÒN BỊ LỌT LƯỚI ({len(leaked_errors)}/{total_errors} LỖI)',
        num_show=min(6, len(leaked_errors)),
        title_color='#ff7f0e',
        save_name='mau_sai_lot_luoi_resnet50_tta.png'
    )
else:
    print('🎉 Tuyệt vời! Không có mẫu lỗi nào bị lọt lưới (0 lỗi lọt lưới).')

# 4. HIỂN THỊ 6 MẪU DỰ ĐOÁN ĐÚNG & GẮP TỰ ĐỘNG AN TOÀN TIÊU BIỂU
correct_auto_samples = auto_samples[auto_samples['true_label'] == auto_samples['pred_label']]
if len(correct_auto_samples) > 0:
    plot_inspected_samples(
        correct_auto_samples.sample(min(6, len(correct_auto_samples)), random_state=SEED),
        title='RESNET-50: 6 MẪU DỰ ĐOÁN ĐÚNG & TỰ ĐỘNG GẮP AN TOÀN TIÊU BIỂU',
        num_show=6,
        title_color='#2ca02c',
        save_name='mau_dung_tu_dong_gap_resnet50_tta.png'
    )